# Pricing fundamentals: instrument JSON, market context, and `ValuationResult`

**Prerequisites:** work through `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, and `01_foundations/market_data_and_curves.ipynb` in this curriculum. This notebook assumes you are comfortable with `DiscountCurve`, `MarketContext`, and basic `finstack_quant` imports.

## Concepts: the instrument pricing pipeline

End-to-end pricing in finstack-quant follows a small, explicit pipeline:

1. **Instrument** — Describe the trade with a canonical `finstack_quant.instrument/1` envelope containing the typed `instrument` payload. Validation rejects bare payloads and returns canonical JSON.
2. **Market** — Provide curves (and optionally FX) inside a **`MarketContext`**, then **serialize it to JSON** so the same snapshot can cross process boundaries or be logged.
3. **Valuation date** — An ISO calendar date (`YYYY-MM-DD`) marking *as of*.
4. **Model** — A string **model key** (for example `discounting` for simple PV off a discount curve).
5. **Result** — The pricer returns a **`ValuationResult`** object directly; read read PV, currency, and any requested **metrics**.

Optional **metrics** are requested by name. The registry knows which measures apply to which instrument; names you request that are not implemented for that instrument are simply omitted from the output.

### Imports

Core entry points live in `finstack_quant.valuations` alongside `ValuationResult`.

In [ ]:
import json

from _shared import instrument_envelope_json

from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import (
    list_models,
    list_models_grouped,
    list_standard_metrics,
    list_standard_metrics_grouped,
    price_instrument,
    price_instrument,
    validate_instrument_json,
)

print("Imported pricing helpers and ValuationResult.")

### Discover metrics: `list_standard_metrics_grouped`

Metrics are organized into groups so you can find what you need quickly. Use `list_standard_metrics_grouped()` to see all metrics by category, or `list_standard_metrics()` for a flat alphabetical list.

In [ ]:
grouped = list_standard_metrics_grouped()
total = sum(len(v) for v in grouped.values())
flat = list_standard_metrics()
print("flat count:", len(flat), "sample:", flat[:3])
print(f"Standard metrics ({total}) across {len(grouped)} groups:\n")
for group, metrics in grouped.items():
    print(f"  {group} ({len(metrics)})")
    for m in metrics:
        print(f"    {m}")

### Validate instrument JSON: `validate_instrument_json`

Pass the raw JSON **string**. The function returns canonical, pretty-printed JSON suitable for `price_instrument`.

In [ ]:
instrument = instrument_envelope_json(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-1",
            "notional": {"amount": "1000000", "currency": "USD"},
            "start_date": "2025-01-15",
            "maturity": "2025-06-15",
            "day_count": "act_360",
            "quote_rate": "0.05",
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)
canonical = validate_instrument_json(instrument)
print(canonical)

### Market context as JSON

Build a `MarketContext`, insert a `DiscountCurve` (id must match the instrument's `discount_curve_id`), then call **`to_json()`** to produce the string passed into pricing.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext

mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75)],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print(f"market_json length: {len(market_json)} chars")
print(market_json[:500] + ("..." if len(market_json) > 500 else ""))

### Price: `price_instrument`

Arguments: canonical instrument JSON, market JSON, **as_of** date string, and optional **`model`** (default `discounting`). The return value is a typed `ValuationResult`; call `.to_json()` for the wire payload.

**PV sign:** A **negative** present value means a **cash outflow** from the perspective of the **position holder** encoded in the instrument (e.g. paying fixed / lending cash in a deposit), consistent with the signed `value.amount` in the JSON envelope.

In [ ]:
result_json = price_instrument(canonical, market_json, "2025-01-15", model="discounting")
print(result_json)

### Price with metrics: `price_instrument`

Pass a list of metric names. **Only metrics registered for that instrument type appear** in `measures` — for example, a deposit may expose `dv01` but not `duration` or `convexity`.

In [ ]:
result_json = price_instrument(
    canonical,
    market_json,
    "2025-01-15",
    model="discounting",
    metrics=["dv01", "duration_mod", "convexity", "bucketed_dv01"],
)
vr = result_json
print("Requested: dv01, duration_mod, convexity, bucketed_dv01")
print("Returned metric keys:", vr.metric_keys())

### Parse results: `ValuationResult`

Use **`from_json`** on the string returned by the pricers. The PV amount is the **`price`** property; **`get_metric(key)`** reads individual scalars.

**PV sign (again):** `ValuationResult.price` uses the same convention — **negative PV** is an **outflow** for the holder of the priced position (compare the deposit example above).

In [ ]:
vr = result_json
print(f"Instrument: {vr.instrument_id}")
print(f"Price: {vr.price:.2f}")
print(f"Currency: {vr.currency}")
print(f"Metric keys: {vr.metric_keys()}")
print(f"Metric count: {vr.metric_count()}")
print(f"Covenants passed: {vr.all_covenants_passed()}")
print("Failed covenants:", vr.failed_covenants())
for key in vr.metric_keys():
    print(f"  {key}: {vr.get_metric(key)}")

### Discover model keys: `list_models` / `list_models_grouped`

Model keys are discovered the same way metrics are — never transcribed from a document. `list_models()` returns the flat list of every model key the pricer registry can dispatch, and `list_models_grouped()` maps each instrument type to the models registered for it.

Both are **registry-derived**, so a key appearing here has a real pricer behind it. A hardcoded list in a notebook, by contrast, drifts silently the moment a model is added or renamed in Rust.

Use `list_models_grouped()` to answer the practical question — *"which models can I pass for this instrument type?"* — before calling `price_instrument(..., model=...)`.

**Metric naming** (see `list_standard_metrics_grouped` earlier for the full catalogue):

- Simple: `dv01`, `duration`, `convexity`, `ytm`
- Bucketed: `bucketed_dv01::USD-OIS::10y`
- Credit-style: `cs01::BOND_A`

In [ ]:
models = list_models()
by_instrument = list_models_grouped()
print(f"Registered model keys ({len(models)}):")
for k in models:
    print(f"  {k}")

print(f"\nInstrument types with a registered pricer: {len(by_instrument)}")
print("Models available per instrument type (a representative slice):")
for instrument in ("deposit", "bond", "interest_rate_swap", "swaption", "equity_option", "credit_default_swap", "term_loan"):
    print(f"  {instrument:<15} {by_instrument[instrument]}")

# The `discounting` workhorse used throughout this notebook covers the widest set.
discounting_instruments = sorted(i for i, ms in by_instrument.items() if "discounting" in ms)
print(f"\nInstrument types accepting model='discounting': {len(discounting_instruments)}")
print(f"  {', '.join(discounting_instruments[:12])}, ...")

print("\nExample metric id shapes:")
for e in ["dv01", "duration", "bucketed_dv01::USD-OIS::10y", "cs01::BOND_A"]:
    print(f"  {e}")

## Mini-example: price a deposit (full pipeline)

1. Define instrument JSON  
2. Validate with `validate_instrument_json`  
3. Build `MarketContext`, serialize with `to_json()`  
4. Call `price_instrument` with **`discounting`** and deposit-relevant metrics  
5. Parse `ValuationResult` and print PV plus each metric

In [ ]:
# Everything below reuses the imports from the "Imports" and "Market context" cells:
# json, date, DiscountCurve, MarketContext, ValuationResult, validate_instrument_json,
# price_instrument, price_instrument.
as_of = "2025-01-15"

# 1) Instrument JSON
raw = instrument_envelope_json(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-MINI",
            "notional": {"amount": "1000000", "currency": "USD"},
            "start_date": as_of,
            "maturity": "2025-06-15",
            "day_count": "act_360",
            "quote_rate": "0.05",
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)

# 2) Validate
instrument_json = validate_instrument_json(raw)
print("Step 2 — canonical instrument (first 220 chars):")
print(instrument_json[:220] + "...")

# 3) Market snapshot
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date.fromisoformat(as_of),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (5.0, 0.75)],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
print("Step 3 — market_json ready:", len(market_json), "chars")

# 4) Price (PV only, then with metrics)
pv_only = price_instrument(instrument_json, market_json, as_of, model="discounting")
print("Step 4a — PV:", pv_only.price, pv_only.currency)

metrics = ["dv01", "deposit_par_rate", "quote_rate", "yf"]
priced = price_instrument(
    instrument_json,
    market_json,
    as_of,
    model="discounting",
    metrics=metrics,
)

# 5) Parse envelope
vr = priced
print("Step 5 — ValuationResult:")
print(f"  id={vr.instrument_id!r}  pv={vr.price:.2f} {vr.currency}")
print(f"  metrics requested: {metrics}")
print(f"  metrics returned: {vr.metric_keys()}")
for key in sorted(vr.metric_keys()):
    print(f"    {key}: {vr.get_metric(key)}")

### Turning a `ValuationResult` into a DataFrame

Every `ValuationResult` exposes `to_dataframe()`, which emits a single-row `pandas.DataFrame` with `instrument_id`, `as_of_date`, `pv`, `currency`, and one column per computed measure. Concatenate across instruments with `pd.concat` to build a portfolio-wide table; print with the built-in `DataFrame.to_string()` renderer so the notebook has no optional display dependency.


In [ ]:
import pandas as pd

# Reuse the `vr` ValuationResult priced above (DEP-MINI with dv01, par rate, etc.)
single_df = vr.to_dataframe()
print("Single-result DataFrame:")
print(single_df.to_string(index=False))

# Portfolio-style roll-up: price a second instrument and concatenate.
second_raw = instrument_envelope_json(
    {
        "type": "deposit",
        "spec": {
            "id": "DEP-MINI-2",
            "notional": {"amount": "500000", "currency": "USD"},
            "start_date": as_of,
            "maturity": "2025-09-15",
            "day_count": "act_360",
            "quote_rate": "0.045",
            "discount_curve_id": "USD-OIS",
            "attributes": {},
        },
    }
)
second_priced = price_instrument(
    validate_instrument_json(second_raw),
    market_json,
    as_of,
    model="discounting",
    metrics=["dv01", "deposit_par_rate", "yf"],
)
vr2 = second_priced

portfolio_df = pd.concat(
    [vr.to_dataframe(), vr2.to_dataframe()],
    ignore_index=True,
)
print("\nPortfolio-wide DataFrame:")
print(portfolio_df.to_string(index=False))


## Takeaways

- A canonical **instrument envelope** plus **`validate_instrument_json`** gives you a stable, loader-ready payload.
- **`MarketContext.to_json()`** produces the market snapshot string expected by **`price_instrument`** / **`price_instrument`**.
- Choose a **model key** (`discounting` is the workhorse for curve-discounted cashflows) — and discover the valid keys with **`list_models`** / **`list_models_grouped`** rather than hardcoding them.
- **`ValuationResult`** is returned directly by the pricers and is the ergonomic read API for PV, currency, covenant flags, and **`measures`**. Call **`.to_json()`** when a pipeline needs the wire payload.
- **Metrics are instrument-specific** — use `list_standard_metrics_grouped` to browse by category; irrelevant IDs are omitted rather than erroring.
- Both catalogues (models and metrics) are read out of the Rust registry at runtime, so the notebook cannot drift from what the library actually supports.

**Next:** continue with `02_pricing/pricing_across_asset_classes.ipynb` or an asset-class deep dive under `02_pricing/instruments/` using the same JSON -> market -> model pattern.